# 1.2 Variant class and 16-gene summary bargraphs

This notebook summarizes variant counts across sequencing runs using the summary Excel files created earlier in the QC workflow.

It makes:

- mean variant class counts by run;
- mean variant counts for the 16 target breast-cancer genes by run;
- optional caller-specific variant class plots.

This notebook is for run-level summary plotting only. Per-variant VAF, alt-depth, and target-status plots are handled separately.
    

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

summary_dir = Path("/home/donetski/Notebooks/OutputFiles/01_gc_noncoding_summary_per_run")
coverage_folder = "all_samples"
# options:
# "all_samples"
# "mean_cov_gt_1x"
# "mean_cov_gt_20x"
# "mean_cov_gt_30x"

output_dir = Path("/home/donetski/Notebooks/OutputFiles/NEW-1.2_variant_gene_specific_bargraphs") / coverage_folder
figure_dir = output_dir / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)

output_prefix = "1.2"
#runs_to_process = "all"
runs_to_process = "run2"
all_runs = ["run1", "run2", "run3", "run4"]

combine_mode = "average" # options: "average" or "sum"
variant_classes = ["SNV", "insertion", "deletion", "indel", "substitution"]
TARGET_GENES = ["ATM", "BARD1", "BRCA1", "BRCA2", "CDH1", "CDKN2A", "CHEK2", "MLH1", "MSH2", "MSH6", "PALB2", "PMS2", "PTEN", "RAD51C", "RAD51D", "TP53"]

In [ ]:
tick_fontsize = 15
axis_label_fontsize = 18
title_fontsize = 20
legend_fontsize = 12

axis_labelpad = 8
title_pad = 10

show_plots = True

print("Input file:", input_csv)
print("Output folder:", output_dir)

## Load run summary tables

Each run summary workbook provides the variant-class summary and the 16-gene summary table.

In [ ]:
def resolve_runs(runs_to_process):
    if isinstance(runs_to_process, str) and runs_to_process.lower() == "all":
        return all_runs
    if isinstance(runs_to_process, str):
        return [runs_to_process.lower()]
    return [run.lower() for run in runs_to_process]

def combined_mean(df):
    if combine_mode == "average":
        return (df["deepvariant_mean"] + df["mutect2_mean"]) / 2
    elif combine_mode == "sum":
        return df["deepvariant_mean"] + df["mutect2_mean"]
    else:
        raise ValueError("combine_mode must be 'average' or 'sum'")

In [ ]:
runs = resolve_runs(runs_to_process)
variant_class_by_run = {}
gene_by_run = {}

for run in runs:
    excel_file = summary_dir / run / coverage_folder / f"{run}_complete_variant_summary_{coverage_folder}.xlsx"
    variant_class_by_run[run] = pd.read_excel(excel_file, sheet_name="variant_class_summary")
    gene_by_run[run] = pd.read_excel(excel_file, sheet_name="gene_summary_16")

print("Loaded:", runs)

In [ ]:
gene_by_run

## Build plotting tables

DeepVariant and Mutect2 means are combined using `combine_mode`.

In [ ]:
variant_class_plot_df = pd.concat([
    df.assign(run=run, combined_mean=combined_mean(df))[["run", "Variant.Class", "combined_mean", "deepvariant_mean", "mutect2_mean"]]
    for run, df in variant_class_by_run.items()
], ignore_index=True)

gene_plot_df = pd.concat([
    df.assign(run=run, combined_mean=combined_mean(df))[["run", "Gene", "combined_mean", "deepvariant_mean", "mutect2_mean"]]
    for run, df in gene_by_run.items()
], ignore_index=True)

variant_class_plot_df.to_csv(output_dir / f"{output_prefix}_variant_class_plot_values.csv", index=False)
gene_plot_df.to_csv(output_dir / f"{output_prefix}_gene_plot_values.csv", index=False)

variant_class_plot_df.head()

## Variant class counts by run

This plot compares the combined mean variant counts per sample for each variant class across runs.

In [ ]:
plot_df = (variant_class_plot_df
    .pivot(index="Variant.Class", columns="run", values="combined_mean")
    .reindex(variant_classes)
    .fillna(0)
)

ax = plot_df.plot(kind="bar", figsize=(10, 6))
ax.set_xlabel("Variant class")
ax.set_ylabel("Mean variants per sample" if combine_mode == "average" else "Summed mean variants per sample")
ax.set_title("Variant class counts by run")

out_png = figure_dir / f"{output_prefix}_variant_class_combined_mean_by_run_{coverage_folder}.png"
plt.tight_layout()
plt.savefig(out_png, dpi=300)
plt.show()

print("Saved:", out_png)

## Sixteen target-gene counts by run

This plot compares mean variant counts per sample across the 16 target breast-cancer genes.

In [ ]:
plot_df = (
    gene_plot_df
    .pivot(index="Gene", columns="run", values="combined_mean")
    .reindex(TARGET_GENES)
    .fillna(0)
)

ax = plot_df.plot(kind="bar", figsize=(14, 6))
ax.set_xlabel("Gene")
ax.set_ylabel("Mean variants per sample" if combine_mode == "average" else "Summed mean variants per sample")
ax.set_title("Variant counts across 16 target genes by run")

out_png = figure_dir / f"{output_prefix}_gene_combined_mean_by_run_{coverage_folder}.png"
plt.tight_layout()
plt.savefig(out_png, dpi=300)
plt.show()

print("Saved:", out_png)

## Caller-specific variant class plots

These plots compare DeepVariant and Mutect2 means for each variant class across runs.

In [ ]:
for vc in variant_classes:
    class_df = (
        variant_class_plot_df[variant_class_plot_df["Variant.Class"].eq(vc)]
        .set_index("run")
        .reindex(runs)
        .fillna(0)
    )

    ax = class_df[["deepvariant_mean", "mutect2_mean"]].plot(kind="bar", figsize=(8, 5))
    ax.set_xlabel("Run")
    ax.set_ylabel("Mean variants per sample")
    ax.set_title(f"{vc} counts by caller and run")

    safe_vc = vc.replace(" ", "_").lower()
    out_png = figure_dir / f"{output_prefix}_{safe_vc}_mean_by_caller_and_run_{coverage_folder}.png"

    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.show()

    print("Saved:", out_png)